# 🔄 Basic Agent Workflows wit Microsoft Foundry (Python)

## 📋 Workflow Orchestration Tutorial

Dis notebook dey introduce di powerful **Workflow Builder** capabilities wey Microsoft Agent Framework get. Learn how to create correct, multi-step agent workflows wey fit handle complex business processes and coordinate multiple AI operations smoothly.

> **Migration note:** Dis sample before e reference GitHub Models. GitHub Models don obsolete (e go retire July 2026), na im e now dey use **Microsoft Foundry** through di `FoundryChatClient`, wey dey target di Azure OpenAI **Responses API**.

## 🎯 Learning Objectives

### 🏗️ **Workflow Architecture**
- **Workflow Builder**: Design and arrange correct multi-step processes
- **Event-Driven Execution**: Manage workflow events and state changes
- **Visual Workflow Design**: Create and show workflow structures
- **Microsoft Foundry Integration**: Use AI models inside workflow contexts

### 🔄 **Process Orchestration**
- **Sequential Operations**: Join many agent tasks in logical order
- **Conditional Logic**: Do decision points and branching workflows
- **Error Handling**: Strong error recovery and workflow toughness
- **State Management**: Follow and manage workflow execution state

### 📊 **Enterprise Workflow Patterns**
- **Business Process Automation**: Automate complex organizational workflows
- **Multi-Agent Coordination**: Arrange many specialized agents
- **Scalable Execution**: Design workflows for big company operations
- **Monitoring & Observability**: Follow workflow performance and results

## ⚙️ Prerequisites & Setup

### 📦 **Required Dependencies**

Install di Agent Framework wey get workflow capabilities:

```bash
pip install agent-framework -U
```

### 🔑 **Microsoft Foundry Configuration**

Sign in wit Azure CLI (`az login`) so `AzureCliCredential` go fit authenticate, den set your Microsoft Foundry project details.

**Environment Setup (.env file):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

### 🏢 **Enterprise Use Cases**

**Business Process Examples:**
- **Customer Onboarding**: Multi-step verification and setup workflows
- **Content Pipeline**: Automated content creation, review, and publishing
- **Data Processing**: ETL workflows with AI-powered transformation
- **Quality Assurance**: Automated testing and validation processes

**Workflow Benefits:**
- 🎯 **Reliability**: Sure execution wit error recovery
- 📈 **Scalability**: Fit handle high-volume process automation
- 🔍 **Observability**: Complete audit trails and monitoring
- 🔧 **Maintainability**: Visual design and modular components

## 🎨 Workflow Design Patterns

### Basic Workflow Structure
```mermaid
graph TD
    A[Strat] --> B[Agent Waka 1]
    B --> C{Decision Poin}
    C -->|Beta| D[Agent Waka 2]
    C -->|Wahala| E[Error Handler]
    D --> F[End]
    E --> F
```

**Key Components:**
- **WorkflowBuilder**: Main orchestration engine
- **WorkflowEvent**: Event handling and communication
- **WorkflowViz**: Visual workflow representation and debugging

Make we build your first smart workflow! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
